# Notebook19 — equivalent learned-kernel implementation

Purpose: preserve the frozen18 convex global/local kernel while reducing deterministic local-gather backward overhead measured in the whole JEPA trace. Inputs: original18 tagged operator and model definitions. Outputs: reusable definitions only; no training or pixel access on import. Runtime: under1second for definition cells; no checkpoint created. The per-forward topology is local derived state, reused across four encoder blocks and discarded immediately. Parameter names, private feature RNG, EMA state and predictor remain unchanged. Gate0.5 is the same mathematics; gate0.1 is the sole registered mathematical ablation. Exact-resume validation belongs to the19runner before the detached pilot.

The inverse-gather derivative uses the symmetry of valid original-grid neighborhoods and reverses the row-major offset order. It avoids the deterministic indexed scatter visible as the largest extra operator in the18profile. No classical FMM or universal manifold certificate is claimed.

In [ ]:
"""Same18 learned kernel: scoped geometry reuse and exact deterministic inverse gather.

No new learned parameter or mathematical operation is introduced at gate0.5.
The gate0.1 variant changes only the existing gate initialization.
"""
from pathlib import Path
import copy,json,math,os,types
import nbformat,torch
from torch import nn
from torch.nn import functional as F
_ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_path=_ROOT/'experiments/18_learned_kernel/operator.ipynb';_old=types.ModuleType('fmm19_original18');_old.__file__=str(_path)
for _cell in nbformat.read(_path,4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_path),'exec'),_old.__dict__)
from fmm_jepa.models import VisionEncoder,validate_positions,position_features
BACKEND='learned-simplex-global-exact-local-convex-mixture-v1-optimized-deterministic-inverse-gather'
ALLOWED_RANKS=_old.ALLOWED_RANKS;OPTIONS=_old.OPTIONS
seeded_feature_directions=_old.seeded_feature_directions;feature_sha256=_old.feature_sha256
local_neighbor_indices=_old.local_neighbor_indices;_checked_inputs=_old._checked_inputs;learned_features=_old.learned_features
dense_learned_kernel_reference=_old.dense_learned_kernel_reference;exact_softmax_attention=_old.exact_softmax_attention

class _SymmetricLocalGather(torch.autograd.Function):
    """Transpose of original undirected neighborhood gather without atomic scatter.

For token i and offset delta, an incoming source is neighbor j=i+delta;
its contribution is stored at the reverse edge j->i (offset -delta).
The local row-major offset ordering makes reverse index L-1-offset exact.
The masks remove absent/padded slots. The fixed sum matches the gather transpose.
Topology is recomputed per encoder forward and is never checkpointed or reused
across inputs, devices, grids, samplers or separate model forwards.
"""
    @staticmethod
    def forward(ctx,joined,neighbors,nearvalid):
        B,H,N,D=joined.shape;L=neighbors.shape[-1]
        ctx.save_for_backward(neighbors,nearvalid);ctx.input_shape=joined.shape
        gather=neighbors.flatten(1)[:,None,:,None].expand(B,H,N*L,D)
        return joined.gather(2,gather).reshape(B,H,N,L,D)
    @staticmethod
    def backward(ctx,gradient):
        neighbors,nearvalid=ctx.saved_tensors;B,H,N,D=ctx.input_shape;L=neighbors.shape[-1]
        reverse=torch.arange(L-1,-1,-1,device=neighbors.device)
        incoming=neighbors*L+reverse[None,None,:]
        gather=incoming.flatten(1)[:,None,:,None].expand(B,H,N*L,D)
        values=gradient.reshape(B,H,N*L,D).gather(2,gather).reshape(B,H,N,L,D)
        result=(values*nearvalid[:,None,:,:,None]).sum(3)
        return result,None,None

def _gather_local_kv(k,v,neighbors,nearvalid):
    D=k.shape[-1];gathered=_SymmetricLocalGather.apply(torch.cat((k,v),dim=-1),neighbors,nearvalid)
    return gathered[...,:D],gathered[...,D:]

def learned_kernel_attention(q,k,v,valid,positions=None,query_features=None,key_features=None,
                             feature_temperature=None,gate=None,tau=None,indices=None,grid=None,
                             local_radius=1,return_stats=False,check_inputs=True,topology=None):
    """Exact application of the declared learned kernel mixture in linear memory.

    G_ij=phi_q(i)^Tphi_k(j)/sum_j phi_q(i)^Tphi_k(j).
    L_ij=1[j in original-grid neighborhood(i)] exp(rawq_i rawk_j/tau)
         /sum_local exp(rawq_i rawk_j/tau).
    A_ij=(1-sigmoid(gate_raw_h))*G_ij+sigmoid(gate_raw_h)*L_ij.
    The API gate argument is the actual probability. Both branches are positive,
    row normalized separately, and mixed without approximate-kernel subtraction.
    """
    Q,K,V,Wq,Wk,beta,mix,tau=_checked_inputs(q,k,v,valid,query_features,key_features,feature_temperature,gate,tau,check_inputs)
    B,H,N,D=Q.shape;rank=Wq.shape[1]
    if indices is None or grid is None:raise ValueError('Original indices/grid required')
    neighbors,nearvalid=local_neighbor_indices(indices,valid,grid,local_radius,check_inputs) if topology is None else topology
    phiq,phik=learned_features(Q,K,Wq,Wk,beta,valid)
    summary=phik.transpose(-1,-2)@V;mass=phik.sum(-2)
    numerator=phiq@summary;denominator=(phiq*mass[:,:,None]).sum(-1)
    # Only empty query rows receive the value1. Never alter a valid denominator.
    denominator=torch.where(valid[:,None],denominator,torch.ones_like(denominator))
    global_output=numerator/denominator[...,None]
    local_keys,local_values=_gather_local_kv(K,V,neighbors,nearvalid)
    logits=torch.einsum('bhnd,bhnld->bhnl',Q,local_keys)/tau[None,:,None,None]
    logits=logits.masked_fill(~nearvalid[:,None],-torch.inf)
    logits=torch.where(valid[:,None,:,None],logits,torch.zeros_like(logits))
    local_weights=logits.softmax(-1)*nearvalid[:,None]
    local_output=torch.einsum('bhnl,bhnld->bhnd',local_weights,local_values)
    output=(1-mix[None,:,None,None])*global_output+mix[None,:,None,None]*local_output
    output=torch.where(valid[:,None,:,None],output,torch.zeros_like(output))
    if check_inputs and not bool(torch.isfinite(output).all()):raise FloatingPointError('Nonfinite output; no numerical validity claim')
    if not return_stats:return output
    stats={'algorithm':BACKEND,'rank':int(rank),'head_dimension':int(D),'local_radius':int(local_radius),
           'local_pairs':int(nearvalid.sum()),'feature_elements':2*B*H*N*rank,
           'global_summary_elements':B*H*rank*(V.shape[-1]+1),
           'dense_reference_entries':B*H*N*N,'far_pair_matrix_constructed':False,
           'adaptive_pivots':False,'gather_count':1,'deterministic_inverse_gather_backward':True,'topology_reused_within_encoder_forward':topology is not None,'exact_for_own_kernel':True,'exact_vit_attention':False,
           'kernel_scope':'Convex mixture of normalized learned positive feature kernel and exact normalized original-grid local raw-QK attention',
           'logit_bound':4*math.sqrt(D),'numerical_denominator_clamp':False}
    return output,stats


class LearnedKernelMixer(_old.LearnedKernelMixer):
    def forward_with_topology(self,x,indices=None,valid=None,grid=None,topology=None):
        q,k,v,valid,geo=self._state(x,indices,valid,grid)
        output=learned_kernel_attention(q,k,v,valid,query_features=self.query_features,key_features=self.key_features,
            feature_temperature=geo['feature_temperature'],gate=geo['local_gate'],tau=geo['temperature'],
            indices=indices,grid=grid,local_radius=self.local_radius,check_inputs=False,topology=topology)
        return self.projection(output.transpose(1,2).reshape_as(x))*valid[...,None]
    def forward(self,x,indices=None,valid=None,grid=None):
        return self.forward_with_topology(x,indices,valid,grid)

class CachedTopologyEncoder(VisionEncoder):
    """Fresh local topology belongs only to this invocation; names/state remain15/18."""
    def forward(self,images,indices=None,valid=None):
        if images.ndim!=4 or images.shape[1]!=3:raise ValueError('images must have shape[B,3,H,W]')
        batch,_,height,width=images.shape;patch=self.patch_size
        if height%patch or width%patch:raise ValueError('image dimensions must divide patch_size')
        grid=(height//patch,width//patch);count=grid[0]*grid[1]
        if indices is None:
            if valid is not None:raise ValueError('valid cannot be supplied without indices')
            indices=torch.arange(count,device=images.device).expand(batch,-1)
        if valid is None:valid=torch.ones_like(indices,dtype=torch.bool)
        if indices.device!=images.device or valid.device!=images.device:raise ValueError('images, indices, valid must share a device')
        validate_positions(indices,valid,batch,count)
        raw=F.unfold(images,kernel_size=patch,stride=patch).transpose(1,2)
        gather=indices.clamp(0,count-1).unsqueeze(-1).expand(-1,-1,raw.shape[-1])
        visible_pixels=torch.where(valid.unsqueeze(-1),raw.gather(1,gather),torch.zeros_like(raw.gather(1,gather)))
        x=self.patch_embed(visible_pixels)
        x=(x+position_features(indices.clamp_min(0),grid,self.dim,x.dtype))*valid.unsqueeze(-1)
        radii={block.mixer.local_radius for block in self.blocks}
        if len(radii)!=1:raise ValueError('One frozen local radius per encoder required')
        topology=local_neighbor_indices(indices,valid,grid,next(iter(radii)),check_inputs=False)
        for block in self.blocks:
            x=x+block.mixer.forward_with_topology(block.norm1(x),indices,valid,grid,topology)
            x=x+block.mlp(block.norm2(x));x=x*valid.unsqueeze(-1)
        return self.norm(x)*valid.unsqueeze(-1)

def replace_encoder_mixers(model,options=None):
    options=dict(options or{});seed=int(options.get('feature_seed',180064))
    if len(model.context.blocks)!=len(model.target.blocks):raise ValueError('Online and EMA depths differ')
    for layer,(context,target)in enumerate(zip(model.context.blocks,model.target.blocks)):
        current={**options,'feature_seed':seed+layer}
        context.mixer=LearnedKernelMixer(context.mixer,current);target.mixer=LearnedKernelMixer(target.mixer,current)
        target.mixer.load_state_dict(context.mixer.state_dict())
    model.context.__class__=CachedTopologyEncoder;model.target.__class__=CachedTopologyEncoder
    model.target.requires_grad_(False).eval();return model


In [ ]:
print('Notebook19 learned-kernel definitions ready; no training launched.')